# NWB Inspector - Simple Interactive Viewer

Load and inspect NWB files interactively.

**Usage:**
1. Run Cell 1 to discover NWB files
2. Run Cell 2 (or 3) to load a specific file by index `k`
3. Add your own cells below to inspect the `nwb` object

In [ ]:
# Cell 1: Setup and discover NWB files

import sys
from pathlib import Path

# Add repo to path if needed
repo_root = Path.cwd()
if (repo_root / 'src').exists():
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

# Try loading local config for data path
data_root = Path(r"D:\\analysis\\nwb")
try:
    from local_config import LOCAL_NWB_PATH
    data_root = LOCAL_NWB_PATH
except ImportError:
    pass

# Discover NWB files
nwb_files = sorted(data_root.rglob("*.nwb"))

print(f"Data directory: {data_root}")
print(f"Found {len(nwb_files)} NWB file(s):\n")

if nwb_files:
    for i, f in enumerate(nwb_files):
        print(f"  k={i}: {f.name}")
else:
    print("  No NWB files found!")
    print(f"  Check path: {data_root}")

## Load NWB File

Set `k` to the file index you want to load (0 to 12 for your 13 files).

In [ ]:
# Cell 2: Load NWB file by index

from pynwb import NWBHDF5IO

# Set the file index to load (0 to 12)
k = 0  # <-- Change this to load different files (0, 1, 2, ..., 12)

if not nwb_files:
    raise FileNotFoundError("No NWB files found. Run Cell 1 first.")

if k < 0 or k >= len(nwb_files):
    raise IndexError(f"k={k} is out of range. Available: 0 to {len(nwb_files)-1}")

nwb_path = nwb_files[k]
print(f"Loading: {nwb_path.name}")
print(f"Full path: {nwb_path}")

# Open the NWB file
io = NWBHDF5IO(str(nwb_path), mode='r', load_namespaces=True)
nwb = io.read()

print(f"\n✓ Loaded successfully!")
print(f"  Session ID: {getattr(nwb, 'session_id', 'N/A')}")
print(f"  Subject: {getattr(getattr(nwb, 'subject', None), 'subject_id', 'N/A')}")
print(f"  Description: {getattr(nwb, 'session_description', 'N/A')[:50]}...")

## Basic Inspection Examples

Run these cells or add your own to inspect the `nwb` object.

In [ ]:
# Cell 3: Quick Overview

print("=" * 60)
print("NWB FILE OVERVIEW")
print("=" * 60)

# Basic metadata
print(f"\nIdentifier: {getattr(nwb, 'identifier', 'N/A')}")
print(f"Session start: {getattr(nwb, 'session_start_time', 'N/A')}")
print(f"Institution: {getattr(nwb, 'institution', 'N/A')}")

# Available data modules
print(f"\nAcquisition modules: {list(nwb.acquisition.keys())}")
print(f"Processing modules: {list(nwb.processing.keys())}")
print(f"Intervals: {list(nwb.intervals.keys()) if hasattr(nwb.intervals, 'keys') else 'N/A'}")

# Units and electrodes
if hasattr(nwb, 'units') and nwb.units is not None:
    print(f"\nUnits table: {len(nwb.units)} units")
    print(f"  Columns: {list(nwb.units.colnames)}")
else:
    print("\nNo units table")

if hasattr(nwb, 'electrodes') and nwb.electrodes is not None:
    print(f"\nElectrodes table: {len(nwb.electrodes)} electrodes")
    print(f"  Columns: {list(nwb.electrodes.colnames)}")
else:
    print("\nNo electrodes table")

In [ ]:
# Cell 4: Inspect Trials / Intervals

if hasattr(nwb, 'intervals') and nwb.intervals is not None:
    for name in nwb.intervals.keys():
        table = nwb.intervals[name]
        print(f"\nInterval table: {name}")
        print(f"  Rows: {len(table)}")
        print(f"  Columns: {list(table.colnames)}")
        
        # Show first few rows as DataFrame
        df = table.to_dataframe().head(3)
        print(f"\n  First 3 rows:")
        display(df)
else:
    print("No intervals found")

In [ ]:
# Cell 5: Inspect Electrodes (channels)

if hasattr(nwb, 'electrodes') and nwb.electrodes is not None:
    elec_df = nwb.electrodes.to_dataframe()
    
    print(f"Electrodes: {len(elec_df)} total\n")
    
    # Group by location (area) if available
    if 'location' in elec_df.columns:
        print("Channels by area:")
        print(elec_df['location'].value_counts())
    
    # Show first few
    print(f"\nFirst 5 electrodes:")
    display(elec_df.head())
else:
    print("No electrodes table found")

In [ ]:
# Cell 6: Inspect Units

if hasattr(nwb, 'units') and nwb.units is not None:
    units_df = nwb.units.to_dataframe()
    
    print(f"Units: {len(units_df)} total\n")
    
    # Show columns
    print(f"Available columns: {list(units_df.columns)}\n")
    
    # Show first few units
    print("First 5 units:")
    display(units_df.head())
    
    # If area/location info available
    if 'electrode_group' in units_df.columns:
        print(f"\nUnits by electrode group:")
        print(units_df['electrode_group'].value_counts())
else:
    print("No units table found")

In [ ]:
# Cell 7: Inspect Raw Signals (Acquisition)

print(f"Acquisition objects: {list(nwb.acquisition.keys())}\n")

for name in nwb.acquisition.keys():
    obj = nwb.acquisition[name]
    print(f"\n{name}:")
    print(f"  Type: {type(obj).__name__}")
    print(f"  Shape: {obj.data.shape if hasattr(obj, 'data') else 'N/A'}")
    print(f"  Rate: {getattr(obj, 'rate', getattr(obj, 'starting_time', 'N/A'))}")
    
    # Show first 100 samples if small enough
    if hasattr(obj, 'data') and len(obj.data.shape) == 1:
        print(f"  First 5 values: {obj.data[:5]}")
    elif hasattr(obj, 'data') and len(obj.data.shape) == 2:
        print(f"  First channel, first 5 values: {obj.data[:5, 0]}")

In [ ]:
# Cell 8: Close file when done (IMPORTANT!)

# Run this when finished inspecting to close the HDF5 file
if 'io' in globals():
    io.close()
    print("✓ NWB file closed")
else:
    print("No file to close")

---

## Custom Inspection

Add your own cells below to explore specific aspects of the data.

**The `nwb` object is your entry point.** Common attributes:
- `nwb.acquisition` - raw signals (LFP, MUAe, etc.)
- `nwb.processing` - processed data
- `nwb.intervals` - trial/epoch information
- `nwb.units` - spike-sorted units
- `nwb.electrodes` - channel/electrode metadata